# Ground Truth Evaluations

Single entry point for running ground-truth camera pose evaluations and comparing results across conditions and datasets.

Compute is handled by `evals/eval_gt.py` (CLI/tmux only — never run directly in a notebook). This notebook loads pre-existing results from `evals/results/` for inspection and comparison.

**Run from repo root** (`/workspace/collab-splats`) so relative paths resolve correctly.

---

## §1 — What are we measuring?

Each pipeline takes a sequence of images and predicts a **camera pose** (position + orientation) for every frame. We compare against **ground-truth poses** recorded by the dataset (e.g. a motion capture system or RGB-D sensor).

### Metrics

| Metric | Plain English | Units | Better = |
|--------|--------------|-------|----------|
| **ATE RMSE** | "On average, how far off was the predicted camera position?" After optimal rigid alignment between predicted and GT trajectories. | metres | Lower |
| **RPE trans RMSE** | "Between consecutive frames, how much does predicted motion drift from true motion?" Captures local drift. | metres | Lower |
| **AUC@30** | "What fraction of frames had position error under 30 cm?" | % (0–100) | Higher |

### Conditions

| Condition | What it runs | When to use |
|-----------|-------------|-------------|
| `baseline` | VGGT-X feedforward only — no refinement | Fastest; use as starting point |
| `ba` | `baseline` + bundle adjustment (2048 tracks, 5 query frames) | Default improvement step |
| `ba_track-density-{N}` | `baseline` + BA with N feature tracks per frame | When `ba` leaves residual drift; slower |
| `lc` | Full loop closure — corrects accumulated drift via pose graph | Long sequences where camera loops back |

### When to use `submap_size`

VGGT-X loads all frames into GPU at once. Sequences longer than ~200 frames exceed GPU memory. Pass `--submap_size 50` to process in overlapping windows of 50 frames.

---

## §2 — Running an evaluation

Compute runs via the CLI — **do not call `run_eval()` inside this notebook** (OOM + slow). Use the terminal instead:

```bash
# Run from /workspace/collab-splats
/opt/conda/envs/nerfstudio/bin/python evals/eval_gt.py --help
/opt/conda/envs/nerfstudio/bin/python evals/eval_gt.py \
    --dataset co3dv2 \
    --seq_dir /data/co3dv2/apple/110_13051_23361 \
    --conditions baseline ba ba_track-density-4096 \
    --submap_size 50
```

Results are written to `evals/results/{dataset}/{seq_name}/run-{timestamp}/`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
%matplotlib inline

In [3]:
# run_eval() is a CLI tool — run from terminal, not from this notebook (OOM + slow).
# See the §2 markdown cell above for the shell command.
#
# /opt/conda/envs/nerfstudio/bin/python evals/eval_gt.py --help
# Results should already exist in evals/results/

In [4]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────
RESULTS = Path("../../evals/results")

print(f"Results dir: {RESULTS}")
print(f"Exists: {RESULTS.exists()}")

Results dir: ../../evals/results
Exists: False


---

## §3 — Load and compare results

`load_results()` discovers all completed runs under `evals/results/`. `compare_runs()` renders a summary DataFrame — call with no arguments to compare all runs, or pass specific run directories for a subset. Columns: run, condition, ATE RMSE, RPE trans, AUC@30, runtime.

In [5]:
def load_results(
    results_root: Path = RESULTS,
    dataset: str | None = None,
    seq_name: str | None = None,
) -> dict[str, dict]:
    """Discover all run-{YYYYMMDD-HHMMSS} directories containing metrics.json.

    Returns {"{dataset}/{seq_name}/run-{timestamp}": metrics_dict}.
    Optionally filter by dataset and/or seq_name.
    """
    root = Path(results_root)
    results = {}
    # Walk results tree; each run dir contains metrics.json
    for metrics_file in sorted(root.glob("*/*/run-*/metrics.json")):
        run_dir = metrics_file.parent
        ds = run_dir.parts[-3]
        seq = run_dir.parts[-2]
        if dataset is not None and ds != dataset:
            continue
        if seq_name is not None and seq != seq_name:
            continue
        key = f"{ds}/{seq}/{run_dir.name}"
        results[key] = json.loads(metrics_file.read_text())
    return results


def compare_runs(
    run_dirs: list[str | Path] | None = None,
    results_root: Path = RESULTS,
) -> pd.DataFrame:
    """Return a DataFrame summarising ATE / RPE / AUC / runtime per run and condition.

    If run_dirs is None, discovers all runs under results_root.
    Pass a single-element list to inspect one run: compare_runs([run_dir]).
    """
    # Load entries from explicit list or full discovery
    if run_dirs is not None:
        entries = {}
        for rd in run_dirs:
            rd = Path(rd)
            mf = rd / "metrics.json"
            if not mf.exists():
                raise FileNotFoundError(f"metrics.json not found in {rd}")
            key = "/".join(rd.parts[-3:])
            entries[key] = json.loads(mf.read_text())
    else:
        entries = load_results(results_root)

    # Flatten per-condition metrics into rows
    rows = []
    for run_key, metrics in entries.items():
        for cond, m in metrics.items():
            rows.append({
                "run": run_key,
                "condition": cond,
                "ATE RMSE (m)": m.get("ate", {}).get("rmse"),
                "RPE trans (m)": m.get("rpe", {}).get("trans_rmse"),
                "AUC@30 (%)": m.get("auc_30"),
                "time (s)": m.get("time_s"),
            })
    df = pd.DataFrame(rows)
    return df.sort_values(["run", "condition"]).reset_index(drop=True)

In [6]:
# Load all results and display a summary table; guard against missing results dir
if RESULTS.exists() and any(RESULTS.glob("**/metrics.json")):
    df = compare_runs()
    display(df)
else:
    print(f"No results yet — run eval_gt.py first (see §2 above).")
    print(f"Expected: {RESULTS}")

No results yet — run eval_gt.py first (see §2 above).
Expected: ../../evals/results


---

## §4 — Trajectory visualisation

`plot_trajectories()` renders a 3D plot comparing the GT trajectory (black) against each condition's predicted trajectory. Pass a run directory path returned by `run_eval()` or discovered via `load_results()`.

In [7]:
_COND_COLORS = {
    "gt": "black",
    "baseline": "tab:red",
    "ba": "tab:blue",
    "lc": "tab:green",
}


def _color_for(cond: str) -> str:
    """Return a consistent color for a given condition name."""
    return _COND_COLORS.get(cond, "tab:purple")


def _cam_positions(poses: np.ndarray) -> np.ndarray:
    """Convert (N,4,4) world-to-cam matrices to (N,3) camera positions in world."""
    R = poses[:, :3, :3]
    t = poses[:, :3, 3]
    return np.einsum("nij,nj->ni", R.transpose(0, 2, 1), -t)


def plot_trajectories(
    run_dir: str | Path,
    conditions: list[str] | None = None,
) -> None:
    """3D trajectory comparison: GT (black) vs each predicted condition.

    Args:
        run_dir:    Path to a run directory containing trajectories.npz.
        conditions: Subset of conditions to plot. None = all available.
    """
    run_dir = Path(run_dir)
    npz = np.load(run_dir / "trajectories.npz", allow_pickle=False)

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    # Plot ground-truth trajectory
    gt_pos = _cam_positions(npz["gt"])
    ax.plot(gt_pos[:, 0], gt_pos[:, 1], gt_pos[:, 2],
            label="gt", color="black", linewidth=2)

    # Plot each predicted condition
    available = [k[5:] for k in npz.files if k.startswith("pred_")]
    to_plot = conditions if conditions is not None else available
    for cond in to_plot:
        key = f"pred_{cond}"
        if key not in npz.files:
            print(f"Warning: {key} not in trajectories.npz — skipping")
            continue
        pos = _cam_positions(npz[key])
        ax.plot(pos[:, 0], pos[:, 1], pos[:, 2],
                label=cond, color=_color_for(cond), linewidth=1)

    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Z (m)")
    ax.set_title(f"Camera Trajectory — {run_dir.name}")
    ax.legend()
    plt.tight_layout()
    plt.show()

---

## §5 — Per-frame ATE

`plot_ate_per_frame()` shows how position error accumulates frame-by-frame. Spikes reveal where a condition drifts — useful for diagnosing where loop closure or denser BA tracks make the most difference.

In [8]:
def plot_ate_per_frame(
    run_dir: str | Path,
    conditions: list[str] | None = None,
) -> None:
    """Line chart of per-frame ATE for each condition.

    Args:
        run_dir:    Path to a run directory containing trajectories.npz.
        conditions: Subset of conditions to plot. None = all available.
    """
    run_dir = Path(run_dir)
    npz = np.load(run_dir / "trajectories.npz", allow_pickle=False)

    available = [k[len("ate_per_frame_"):] for k in npz.files if k.startswith("ate_per_frame_")]
    to_plot = conditions if conditions is not None else available

    fig, ax = plt.subplots(figsize=(12, 4))
    for cond in to_plot:
        key = f"ate_per_frame_{cond}"
        if key not in npz.files:
            print(f"Warning: {key} not in trajectories.npz — skipping")
            continue
        per_frame = npz[key]
        rmse = float(np.sqrt(np.mean(per_frame ** 2)))
        ax.plot(per_frame, label=f"{cond} (RMSE={rmse:.3f}m)", color=_color_for(cond))

    ax.set_xlabel("Frame")
    ax.set_ylabel("ATE (m)")
    ax.set_title(f"Per-frame Absolute Trajectory Error — {run_dir.name}")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [9]:
# Replace with an actual run directory path
# run_dir = "evals/results/co3dv2/apple/run-20260520-143201"
# plot_trajectories(run_dir)
# plot_ate_per_frame(run_dir)